# ImmoAnalyse — Module 2 : préparation des données

Ce notebook exécute le workflow étape par étape en réutilisant les fonctions de `src/`.

In [ ]:
# Le notebook est dans notebooks/ : on ajoute la racine du projet au chemin Python
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

import pandas as pd
from src.config import *
from src.data_loader import load_dataset
from src.audit import audit_dataset
from src.eda import run_eda
from src.anomalies import handle_anomalies
from src.feature_engineering import engineer_features
from src.preprocessing import split_data, build_preprocessor
from src.validation import validate_transformed_data

## 1. Chargement et audit

In [ ]:
df = load_dataset(DATA_PATH)
report = audit_dataset(df, AUDIT_DIR)
print(report["n_rows"], "lignes,", report["n_columns"], "colonnes,",
      report["duplicate_ids"], "identifiants dupliqués")
df.head()

## 2. Analyse exploratoire (figures enregistrées dans `reports/figures/`)

In [ ]:
run_eda(df, TARGET_COLUMN, FIGURES_DIR)

## 3. Traitement des anomalies
Chaque décision (corriger / exclure / conserver) est justifiée.

In [ ]:
df, decisions = handle_anomalies(df, AUDIT_DIR)
pd.DataFrame(decisions)[["anomalie", "action", "lignes_concernees"]]

## 4. Feature engineering

In [ ]:
df = engineer_features(df)
df[["surface_per_room", "listing_year", "listing_month", "is_dakar"]].describe()

## 5. Split train / validation / test (stratifié par transaction)

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)
print(len(X_train), len(X_val), len(X_test))
print(X_train["transaction"].value_counts(normalize=True).round(3).to_dict())
print(X_test["transaction"].value_counts(normalize=True).round(3).to_dict())

## 6. Pipeline : fit sur train uniquement, transform sur validation et test

In [ ]:
preprocessor = build_preprocessor(X_train)
Xt_train = preprocessor.fit_transform(X_train)   # apprentissage sur train seulement
Xt_val = preprocessor.transform(X_val)           # transform uniquement
Xt_test = preprocessor.transform(X_test)
Xt_train.shape, Xt_val.shape, Xt_test.shape

## 7. Validation qualité

In [ ]:
n = Xt_train.shape[1]
[validate_transformed_data(m, name, n) for m, name in
 [(Xt_train, "train"), (Xt_val, "validation"), (Xt_test, "test")]]

## 8. Export
L'export complet (matrices, pipeline `.joblib`, rapports) est fait par :

```bash
python scripts/run_module_02.py
```